## 🏗️ LeetCode 74: Search a 2D Matrix
---

<div style="padding: 15px; border-left: 8px solid #9C27B0; background-color: #f3e5f5; color: #4a148c; border-radius: 4px;">
    <strong>The Core Insight:</strong> This matrix is one sorted array folded into a 2D grid. If you unfold it back into 1D, standard binary search applies. The key is the index conversion: row = mid // cols, col = mid % cols.
</div>

### 🛠️ The Mathematical Model

A matrix with `rows × cols` elements has indices 0 to `rows*cols - 1` in 1D. Element at 1D index `i` sits at `matrix[i // cols][i % cols]` in 2D. Binary search runs on the 1D index space.

$$\text{1D index} \rightarrow \text{row} = \lfloor i / \text{cols} \rfloor, \quad \text{col} = i \bmod \text{cols}$$

---

### 📋 Problem

Write an efficient algorithm that searches for a value `target` in an `m × n` integer matrix where each row is sorted ascending and the first integer of each row is greater than the last integer of the previous row.

**Example 1:**
```
Input:  matrix = [[1,3,5,7],[10,11,16,20],[23,30,34,60]], target = 3
Output: true
```

**Example 2:**
```
Input:  matrix = [[1,3,5,7],[10,11,16,20],[23,30,34,60]], target = 13
Output: false
```

**Constraints:** m, n ∈ [1, 100] | -10⁴ ≤ matrix[i][j], target ≤ 10⁴

### 🧠 Choose Your Mental Model

<table style="width:100%; border-collapse: collapse;">
    <tr style="background-color: #f2f2f2; text-align: left;">
        <th style="padding: 12px; border: 1px solid #ddd;">Model</th>
        <th style="padding: 12px; border: 1px solid #ddd;">The "Story"</th>
        <th style="padding: 12px; border: 1px solid #ddd;">Mechanism</th>
    </tr>
    <tr>
        <td style="padding: 12px; border: 1px solid #ddd;"><b>Unfolded Array</b></td>
        <td style="padding: 12px; border: 1px solid #ddd;">"Imagine cutting the matrix row by row and laying the rows end-to-end — you get a sorted array. Binary search that array, then fold the index back into 2D coordinates."</td>
        <td style="padding: 12px; border: 1px solid #ddd;">mid // cols = row, mid % cols = col — the fold/unfold math</td>
    </tr>
    <tr>
        <td style="padding: 12px; border: 1px solid #ddd;"><b>1D Abstraction</b></td>
        <td style="padding: 12px; border: 1px solid #ddd;">"I don't see rows and columns. I see positions 0 to m*n-1. The 2D structure is just a display format."</td>
        <td style="padding: 12px; border: 1px solid #ddd;">Binary search on integer range [0, rows*cols-1], translate at access time</td>
    </tr>
</table>

---

### ⚠️ Performance Warning

<div style="padding: 10px; border: 1px solid #ffe58f; background-color: #fffbe6; border-radius: 4px;">
    <strong>Note:</strong> A nested loop scans every cell — O(m × n). Binary search treats the matrix as a 1D sorted array and runs in O(log(m × n)).
</div>

## 🐢 Approach 1: Brute Force — $O(m \times n)$

In [ ]:
def searchMatrix_brute(matrix, target):
    """
    Brute Force: Check every cell
    Time: O(m * n) | Space: O(1)
    """
    for row in matrix:
        for val in row:
            if val == target:
                return True
    return False


print(searchMatrix_brute([[1,3,5,7],[10,11,16,20],[23,30,34,60]], 3))   # True
print(searchMatrix_brute([[1,3,5,7],[10,11,16,20],[23,30,34,60]], 13))  # False

## 🔬 Complexity Analysis: $O(m \times n)$ vs. $O(\log(m \times n))$

<div style="padding: 15px; border-left: 8px solid #2196F3; background-color: #f0f7ff; color: #0d47a1; border-radius: 4px;">
    <strong>The Core Insight:</strong> The matrix's row-ordering property means the entire grid is a sorted 1D sequence in disguise. Once we recognize this, we can apply standard binary search on the index range [0, m*n-1], converting 1D indices to 2D coordinates at each step.
</div>

---

## 📉 Why Brute Force Fails: The $O(m \times n)$ Trap

A nested loop iterates through every cell without using the sorted property. For a 100×100 matrix, that's 10,000 cell checks. Binary search needs at most log₂(10,000) ≈ 14 checks.

| Input Type | Brute Force Performance | Reason |
|------------|------------------------|--------|
| **Target in last cell** | m × n comparisons | Must scan the entire matrix |
| **Target not present** | m × n comparisons | Cannot short-circuit without sorted property |

---

## 🚀 The Optimal Approach: $O(\log(m \times n))$

Treat the matrix as a 1D sorted array of length `rows × cols`. Binary search on indices 0 to `rows*cols - 1`. For each `mid` index, convert to matrix coordinates: `row = mid // cols`, `col = mid % cols`.

### The Key Lifecycle Rule
1. **Compute mid** in 1D space: `mid = left + (right - left) // 2`
2. **Convert to 2D**: `row = mid // cols`, `col = mid % cols`, access `matrix[row][col]`
3. **Compare and eliminate** exactly like standard binary search

---

## ✅ Mathematical Proof

$$\text{1D size} = m \times n \Rightarrow \text{Binary search steps} = \lceil \log_2(m \times n) \rceil = \lceil \log_2 m + \log_2 n \rceil$$

<div style="padding: 10px; border-left: 8px solid #4CAF50; background-color: #f1f8f4; color: #1b5e20; border-radius: 4px;">
    <strong>✅ Summary:</strong> The 2D structure is irrelevant once we see the matrix as a sorted 1D array. Index math (// and %) converts between representations in O(1). Binary search runs in O(log(m*n)).
</div>

## 🚀 Approach 2: Binary Search on Flattened Index — $O(\log(m \times n))$
---

Instead of nested loops, we use **1D binary search** with a **2D index translation** at each step.

As we iterate:
1. Set `left = 0, right = rows * cols - 1`
2. Compute `mid`, translate to `row = mid // cols, col = mid % cols`
3. Compare `matrix[row][col]` to target — eliminate left or right half

In [ ]:
def searchMatrix(matrix, target):
    """
    Optimal: Binary Search on 1D flattened index
    Time: O(log(m * n)) | Space: O(1)
    """
    rows, cols = len(matrix), len(matrix[0])
    left, right = 0, rows * cols - 1

    while left <= right:
        mid = left + (right - left) // 2
        row, col = mid // cols, mid % cols   # key: unfold 1D index to 2D coordinates
        val = matrix[row][col]

        if val == target:
            return True
        elif val < target:
            left = mid + 1
        else:
            right = mid - 1

    return False


print("Optimal:", searchMatrix([[1,3,5,7],[10,11,16,20],[23,30,34,60]], 3))   # True
print("Optimal:", searchMatrix([[1,3,5,7],[10,11,16,20],[23,30,34,60]], 13))  # False

## 🎤 5 Interview Q&A

### Q1: How do you convert a 1D index to 2D matrix coordinates?

**Answer:** For a matrix with `cols` columns: `row = index // cols`, `col = index % cols`. This is integer division for the row, modulo for the column. Example: in a 4-column matrix, 1D index 7 → row = 7 // 4 = 1, col = 7 % 4 = 3 → matrix[1][3].

---

### Q2: What if the matrix rows are sorted but the first element of each row is NOT greater than the last of the previous row?

**Answer:** Then the matrix is not a contiguous sorted sequence — we can't use 1D binary search. Instead, use the staircase search: start at top-right corner (largest in row 1, smallest in last column). If target < current, move left. If target > current, move down. O(m + n) time.

---

### Q3: What is the time complexity of the optimal approach and why?

**Answer:** $O(\log(m \times n))$. The search space has `m * n` elements. Each step eliminates half. So the number of steps is log₂(m × n). The 2D index conversion (// and %) is O(1), so it doesn't change the asymptotic complexity.

---

### Q4: Could you binary search each row individually instead?

**Answer:** Yes — binary search each of the `m` rows for target: O(m log n). This is worse than O(log(m*n)) = O(log m + log n) which equals O(log(m*n)). The full-matrix binary search is strictly better because it uses the *inter-row* sorted property, not just within-row sorting.

---

### Q5: What are the edge cases to watch for?

**Answer:** (1) 1×1 matrix — `left=right=0`, checks the single element. (2) Single row — same as 1D binary search. (3) Single column — `cols=1`, every `mid // 1 = mid` gives row correctly, `mid % 1 = 0` gives col 0 correctly. All edge cases are handled by the general formula.

## 📚 Key Terminology

| Term | Definition |
|------|------------|
| **Flattening** | Converting a multi-dimensional structure to 1D — a matrix of m×n becomes an array of length m*n |
| **Index Translation** | Converting between 1D and 2D coordinates: `row = i // cols`, `col = i % cols` |
| **Contiguous Sorted** | The matrix property that every row is sorted AND each row's first element exceeds the previous row's last — enables 1D binary search |
| **Staircase Search** | Alternative algorithm starting at top-right: move left if too big, down if too small — O(m+n), used when row-first-element property doesn't hold |
| **Integer Division (//)** | Floor division — used to compute the row in 2D translation |
| **Modulo (%)** | Remainder — used to compute the column in 2D translation |
| **Search Space** | The range of 1D indices [0, m*n-1] considered as binary search candidates |
| **2D Matrix** | An m×n grid — in Python, a list of lists; rows are the outer list, columns the inner |

## 💼 The Citi Narrative

**Context:** Capacity reporting at Citi — daily metric snapshots stored as a 2D structure: rows = servers (6,000), columns = time slots (24 hourly readings). Finding whether a specific (server, hour) reading exceeded a threshold.

**Scenario:** Given a sorted report matrix (sorted by server_id, then by hour within each row), determine if a specific utilization value exists without scanning all 144,000 cells.

**How this pattern applied:** The report matrix satisfied the contiguous-sorted property — server IDs increase row by row, and within each row, hourly readings increase. Binary search on the flattened index found any value in O(log 144,000) ≈ 17 comparisons instead of up to 144,000.

**Impact:** Threshold-existence queries on the daily capacity matrix went from full-table scans to sub-millisecond lookups. This enabled interactive capacity dashboards — analysts could ask "did any server hit 90% CPU at any hour yesterday?" and get an answer in real time.

In [ ]:
# -------------------------------------------------------
# PRACTICE: Implement the STAIRCASE search for a matrix
# where rows are sorted but the inter-row property does NOT hold.
# Start at top-right. If target < current, go left. If target > current, go down.
# -------------------------------------------------------

def searchMatrixII(matrix, target):
    # Your solution here
    pass


# Test
print(searchMatrixII([[1,4,7,11],[2,5,8,12],[3,6,9,16],[10,13,14,17]], 5))    # Expected: True
print(searchMatrixII([[1,4,7,11],[2,5,8,12],[3,6,9,16],[10,13,14,17]], 20))   # Expected: False

## 🎯 Summary: Key Takeaways

### The Pattern
**Binary Search on Flattened Index** — Treat a 2D matrix as a 1D sorted array; translate indices at access time

### When to Use It
- ✅ Matrix where each row is sorted AND row[i][0] > row[i-1][-1] (contiguous sorted)
- ✅ Need O(log(m×n)) instead of O(m×n)
- ❌ **Don't use when:** Matrix is sorted within rows but not between rows — use staircase search instead

### Complexity
| Approach | Time | Space |
|----------|------|-------|
| Brute Force (nested loop) | $O(m \times n)$ | $O(1)$ |
| Optimal (Binary Search on 1D) | $O(\log(m \times n))$ | $O(1)$ |

### Interview Confidence Checklist
- [ ] Can explain why this matrix enables 1D binary search
- [ ] Can write the index translation formula from memory
- [ ] Can explain the staircase alternative and when to use it
- [ ] Can state time and space complexity with justification
- [ ] Can connect to the Citi capacity matrix use case

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Master 2D Binary Search and you've mastered a key insight: structure is just a display format. The algorithm operates on the underlying data. 🚀